# 04 — Behavioural features (Person 4)

Run all cells to generate separate train/test behavioural tables, with one row per TRANSACTION_HASH and no labels. **04 denotes the role, not execution order: 01 cleaning → 02 graph and 04 behaviour → 03 ML/prediction.** Role 03 must run after 04 and retains the stratified 80/20 split. This notebook performs no training or predictions.

Prerequisite, from the repository root: `python notebooks/run_data_cleaning.py`. Person 1's verified fresh-kernel notebook exports train, validation and test under data/processed/.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "behaviour_features.py").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "behaviour_features.py").exists():
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")
sys.path.insert(0, str(ROOT))
from src.generate_behaviour import generate
OUTPUTS = ROOT / "outputs"


## Input mapping and historical rules

The connector verifies Person 1's ACCOUNT, TIMESTAMP and payment JSON against the raw split. It derives the original sender, final recipient, first-leg paid USD and delivered received USD. Calendar features remain Person 1's work.

All windows include the lower boundary and exclude every payment at the current timestamp. Missing historical amounts make sums missing; first outgoing gaps stay missing. Train and test histories remain separate. The 80/20 holdout is applied later by 03 ML; organiser validation/ is not that holdout.


In [ ]:
train_features = generate(
    ROOT / "data/processed/train_cleaned.parquet",
    ROOT / "data/raw/train",
    OUTPUTS / "train_behaviour_features.parquet",
)
train_features.head()


In [ ]:
test_features = generate(
    ROOT / "data/processed/test_cleaned.parquet",
    ROOT / "data/raw/test",
    OUTPUTS / "test_behaviour_features.parquet",
)
test_features.head()


In [ ]:
from src.behaviour_features import FEATURES
for name, frame in [("train", train_features), ("test", test_features)]:
    assert list(frame.columns) == ["TRANSACTION_HASH"] + FEATURES
    assert frame.TRANSACTION_HASH.is_unique
    assert frame.TRANSACTION_HASH.notna().all()
    print(name, len(frame), "rows; six behaviour features; no labels")


## Handoff to 03 ML

Use outputs/train_behaviour_features.parquet and outputs/test_behaviour_features.parquet. Merge each by TRANSACTION_HASH with validate="one_to_one", checking identical hash sets. The existing model integration uses load_for_ml to reject missing, changed or stale outputs. Re-run 04 after regenerating cleaned inputs. Keep the 80/20 stratified random split with random_state=0 in the model. Do not train inside 04.

These are offline features computed from label-free histories within each supplied split before model holdout, matching the agreed starter protocol. Outputs and provenance manifests stay local in ignored outputs/. No data is uploaded. Re-running replaces only these generated files.

For terminal execution: `python -m src.generate_behaviour --train data/raw/train --test data/raw/test`. Both routes call the same tested implementation.
